In [1]:
from random import random


# ─────────────────────────────────────────────
# Función que se quiere minimizar (Six-Hump Camelback)
# ─────────────────────────────────────────────
def funcion(x, y):
    sum1 = x**2 * (4 - 2.1*x**2 + x**4/3.0)
    sum2 = x * y
    sum3 = y**2 * (-4 + 4*y**2)
    return sum1 + sum2 + sum3


# Devuelve un número aleatorio dentro de un rango
def aleatorio(inf, sup):
    return random() * (sup - inf) + inf


# ─────────────────────────────────────────────
# Clase Partícula
# ─────────────────────────────────────────────
class Particula:
    # Parámetros para actualizar la velocidad
    inercia   = 1.4
    cognitiva = 2.0
    social    = 2.0

    # Límites del espacio de soluciones
    infx = -2.0
    supx =  2.0
    infy = -1.0
    supy =  1.0

    # Factor de ajuste de la velocidad inicial
    ajusteV = 100.0

    def __init__(self):
        # Posición inicial aleatoria dentro de los límites
        self.x = aleatorio(Particula.infx, Particula.supx)
        self.y = aleatorio(Particula.infy, Particula.supy)

        # Velocidad inicial aleatoria pequeña
        rangoX = (Particula.supx - Particula.infx) / Particula.ajusteV
        rangoY = (Particula.supy - Particula.infy) / Particula.ajusteV
        self.vx = aleatorio(-rangoX, rangoX)
        self.vy = aleatorio(-rangoY, rangoY)

        # Mejor posición local = posición inicial
        self.xLoc    = self.x
        self.yLoc    = self.y
        self.valorLoc = funcion(self.x, self.y)

    def actualizaVelocidad(self, xGlob, yGlob):
        r1 = random()
        r2 = random()
        # Componente cognitiva (atracción al mejor propio)
        # Componente social    (atracción al mejor global)
        self.vx = (Particula.inercia   * self.vx
                 + Particula.cognitiva * r1 * (self.xLoc - self.x)
                 + Particula.social    * r2 * (xGlob     - self.x))
        self.vy = (Particula.inercia   * self.vy
                 + Particula.cognitiva * r1 * (self.yLoc - self.y)
                 + Particula.social    * r2 * (yGlob     - self.y))

    def actualizaPosicion(self):
        self.x += self.vx
        self.y += self.vy

        # Mantener dentro de los límites (reflexión simple)
        self.x = max(Particula.infx, min(Particula.supx, self.x))
        self.y = max(Particula.infy, min(Particula.supy, self.y))

        # Actualizar mejor posición local si mejora
        valor = funcion(self.x, self.y)
        if valor < self.valorLoc:
            self.xLoc     = self.x
            self.yLoc     = self.y
            self.valorLoc = valor


# ─────────────────────────────────────────────
# Algoritmo principal del enjambre (PSO)
# ─────────────────────────────────────────────
def enjambreParticulas(particulas, iteraciones, reduccionInercia):
    # Inicializar el mejor global con la mejor partícula inicial
    mejorInicial = min(particulas, key=lambda p: p.valorLoc)
    xGlob    = mejorInicial.xLoc
    yGlob    = mejorInicial.yLoc
    valorGlob = mejorInicial.valorLoc

    # Bucle principal de simulación
    for _ in range(iteraciones):
        # Actualizar velocidad y posición de cada partícula
        for p in particulas:
            p.actualizaVelocidad(xGlob, yGlob)
            p.actualizaPosicion()

        # Actualizar mínimo global tras mover todas las partículas
        mejorParticula = min(particulas, key=lambda p: p.valorLoc)
        if mejorParticula.valorLoc < valorGlob:
            xGlob     = mejorParticula.xLoc
            yGlob     = mejorParticula.yLoc
            valorGlob = mejorParticula.valorLoc

        # Reducir la inercia en cada iteración
        Particula.inercia *= reduccionInercia

    return (xGlob, yGlob, valorGlob)


# ─────────────────────────────────────────────
# 1. PROGRAMACIÓN ESTRUCTURADA (fuerza bruta)
# ─────────────────────────────────────────────
def busquedaExhaustiva(pasos=1000):
    """Recorre una malla uniforme y devuelve el mínimo encontrado."""
    infx, supx = -2.0, 2.0
    infy, supy = -1.0, 1.0

    xMin = yMin = None
    valorMin = float('inf')

    for i in range(pasos + 1):
        x = infx + i * (supx - infx) / pasos
        for j in range(pasos + 1):
            y = infy + j * (supy - infy) / pasos
            v = funcion(x, y)
            if v < valorMin:
                valorMin = v
                xMin, yMin = x, y

    return (xMin, yMin, valorMin)


# ─────────────────────────────────────────────
# Ejecución y comparación
# ─────────────────────────────────────────────
if __name__ == "__main__":
    # Parámetros del PSO
    nParticulas  = 10
    iteraciones  = 100
    redInercia   = 0.9

    # --- PSO ---
    particulas = [Particula() for _ in range(nParticulas)]
    xP, yP, vP = enjambreParticulas(particulas, iteraciones, redInercia)

    # --- Búsqueda exhaustiva ---
    xE, yE, vE = busquedaExhaustiva(pasos=1000)

    # --- Comparación ---
    print("=" * 50)
    print("  COMPARACIÓN DE RESULTADOS")
    print("=" * 50)
    print(f"\n{'Método':<30} {'x':>8} {'y':>8} {'f(x,y)':>12}")
    print("-" * 50)
    print(f"{'PSO (enjambre)':<30} {xP:>8.5f} {yP:>8.5f} {vP:>12.8f}")
    print(f"{'Búsqueda exhaustiva':<30} {xE:>8.5f} {yE:>8.5f} {vE:>12.8f}")
    print("-" * 50)

    # Mínimos globales conocidos de la función Six-Hump Camelback:
    # (0.0898, -0.7126) y (-0.0898, 0.7126)  → f ≈ -1.0316
    print("\nMínimos globales teóricos conocidos:")
    print(f"  ( 0.08983, -0.71266) → f = {funcion( 0.08983, -0.71266):.8f}")
    print(f"  (-0.08983,  0.71266) → f = {funcion(-0.08983,  0.71266):.8f}")
    print("=" * 50)

  COMPARACIÓN DE RESULTADOS

Método                                x        y       f(x,y)
--------------------------------------------------
PSO (enjambre)                  0.08984 -0.71266  -1.03162845
Búsqueda exhaustiva            -0.08800  0.71200  -1.03161290
--------------------------------------------------

Mínimos globales teóricos conocidos:
  ( 0.08983, -0.71266) → f = -1.03162845
  (-0.08983,  0.71266) → f = -1.03162845
